# Project 1 Template — Run One Task on 40 Items
**AI Models and Agents for Business · Project 1: Automate One Task**

This notebook is the measuring instrument for Part B of your project. You do **not** need to write code. You will edit the cells marked **✏️ CHANGE ME**, run everything top to bottom, and carry the numbers into your memo.

### What this notebook does
1. Loads your 40 items (or a 5-item sample so you can see it work first).
2. Sends each item to an AI model with *your* task description and collects a structured answer.
3. Saves a results file with 20 randomly chosen items marked for **hand-labeling**.
4. After you label them, reports **accuracy, the most common error, and cost**.
5. Lets you change one thing (your prompt), re-run, and see whether it helped.

### The rule of this project
**Hand-label before you report any number.** The model does not get to grade itself. If your hand-labels and the model's answers disagree, your labels win.

### One-time setup: store your key in Colab Secrets
1. Click the **🔑 key icon** in the left sidebar of Colab.
2. Click **Add new secret**. Name: `GROQ_API_KEY`. Value: your key (it starts with `gsk_`).
3. Turn on **Notebook access** for this notebook.

Colab will remember the key for all your notebooks, so you never paste it into a cell. If you skip this, the setup cell will simply ask you to paste the key each time (the box hides what you type).

In [ ]:
# Run this cell first. Do not modify.
!pip install groq --quiet

import time, json, textwrap
from groq import Groq
import groq

def _get_key():
    """Look for the key in Colab Secrets first; otherwise ask for it (hidden as you type)."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

API_KEY = _get_key()
client = Groq(api_key=API_KEY)
MODEL = "openai/gpt-oss-20b"         # if Groq retires this name, change it here only

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

def _retry_after(err, default=15):
    """Seconds the free tier asks us to wait (from the error's retry-after header), capped at 60."""
    try:
        return min(60.0, float(err.response.headers.get("retry-after", default)))
    except Exception:
        return float(default)

def ask(prompt, system=None, temperature=0.0, max_tokens=300, messages=None, model=None):
    """Send one request to the model and return the full response object.
    Waits and retries if the free tier says 'slow down'."""
    if messages is None:
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    for attempt in range(6):
        try:
            return client.chat.completions.create(model=model or MODEL, messages=messages,
                                                  temperature=temperature, max_tokens=max_tokens)
        except groq.RateLimitError as err:
            wait = _retry_after(err)
            print(f"(rate limit hit - waiting {wait:.0f} seconds, then trying again)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited. Wait a minute and re-run this cell.")

def say(prompt, **kw):
    """Same as ask(), but returns just the text of the answer."""
    return (ask(prompt, **kw).choices[0].message.content or "").strip()

print("✅ Setup complete. Model:", MODEL)

## Part 1 — ✏️ Configure your run
Everything you need to change is in the next cell. **Run it as is first** to see the sample work; then replace the settings with your own.

* `DATASET` picks a starter task description from the Workflow Library. If you are using your own workflow, write your own `TASK_DESCRIPTION`.
* `USE_SAMPLE = True` runs on 5 built-in sample emails. Set it to `False` to upload your own CSV (it needs an `id` column; every other column is shown to the model).
* `SAMPLE_SIZE` is how many items to run (40 for the project).
* Prices are placeholders so that you can estimate what a *firm* would pay. The free tier costs you nothing, but a business would pay list price. **Check the current prices on Groq's pricing page** and update both numbers.

In [ ]:
# ✏️ CHANGE ME ---------------------------------------------------------
DATASET        = "support_email_triage"   # support_email_triage, product_reviews, expense_categorization,
                                          # meeting_notes, lead_qualification, job_posting_skills, or "my_own"
USE_SAMPLE     = True        # True = 5 built-in sample items. False = upload your own CSV in Part 2.
SAMPLE_SIZE    = 40          # how many items to run
SEED           = 495         # keeps your random 20 the same each time you re-run
N_TO_LABEL     = 20          # how many outputs your team will hand-label

# What a firm would pay (dollars per MILLION tokens). Placeholders: look up current prices and update.
PRICE_IN_PER_M  = 0.075
PRICE_OUT_PER_M = 0.30

# For the economics in Part 7
MONTHLY_VOLUME  = 2000       # how many items does the real workflow handle per month?
# ----------------------------------------------------------------------

DATASET_STARTERS = {
 "support_email_triage": (
   "You triage customer-support emails for an outdoor-gear retailer.\n"
   "Categories: Billing, Shipping, Technical, Returns, Account, Other. Pick the customer's MAIN request.\n"
   "Urgency: high = a deadline within days, a safety issue, or suspected fraud/account takeover; "
   "medium = action needed but no hard deadline; low = a question or feedback.\n"
   'Return ONLY JSON like {"category": "...", "urgency": "..."}'),
 "product_reviews": (
   "You read product reviews for an online store.\n"
   "Sentiment: positive, neutral, or negative. Main topic: the aspect that drives the reviewer's opinion; "
   "one of quality, price, shipping, support, usability.\n"
   'Return ONLY JSON like {"sentiment": "...", "main_topic": "..."}'),
 "expense_categorization": (
   "You categorize business expenses.\n"
   "Categories: Travel, Meals, Software, Office Supplies, Marketing, Training, Other.\n"
   'Return ONLY JSON like {"category": "..."}'),
 "meeting_notes": (
   "You extract action items from meeting notes. An action item is a FUTURE task that a named person "
   "commits to or is assigned. Past work, hypothetical ideas, cancelled plans, and tasks with no named owner do not count.\n"
   'Return ONLY JSON like {"action_items": ["Owner: task", ...], "n_action_items": 0}'),
 "lead_qualification": (
   "You qualify sales leads from inquiry emails.\n"
   "Hot = budget stated or clearly approved AND decision timeline within 60 days AND the writer is a decision-maker "
   "or has one involved. Warm = genuine interest but missing one of those three. "
   "Cold = vague, student or researcher, competitor fishing, or no buying signal.\n"
   'Return ONLY JSON like {"label": "hot|warm|cold", "reason": "one sentence"}'),
 "job_posting_skills": (
   "You extract required skills from job postings. Include ONLY skills the posting states as required or must-have. "
   "Do not include preferred, bonus, plus, or nice-to-have skills. Degrees, years of experience and physical "
   "requirements are not skills.\n"
   'Return ONLY JSON like {"required_skills": ["...", "..."]}'),
 "my_own": "Describe your task here, define the allowed answers, and end with: Return ONLY JSON like {...}",
}

# Your task description. Start from the starter, then make it better. This is the thing you are testing.
TASK_DESCRIPTION = DATASET_STARTERS[DATASET]

print("Task description the model will see:\n")
print(TASK_DESCRIPTION)

## Part 2 — Load the items
If `USE_SAMPLE` is `True`, this uses five built-in emails. Otherwise it asks you to upload your CSV. The six Workflow Library datasets are in the `data/p1_workflow_library/` folder of this course's GitHub repository (`itao495-notebooks`): download the one you want to your computer, then upload it here.

In [ ]:
import pandas as pd, io, random, math, re

SAMPLE_ITEMS = pd.DataFrame([
 {"id": "S-1", "subject": "Package says delivered but it is not here",
  "body": "Tracking says my boots were delivered yesterday at 2pm but nothing is at my door. I checked with my neighbors."},
 {"id": "S-2", "subject": "Charge I do not recognize!!",
  "body": "There is a $64.20 charge on my card from your company and I have not ordered in over a year. I think someone has my card. I already froze it."},
 {"id": "S-3", "subject": "Wrong size",
  "body": "The rain shell is too tight in the shoulders. Tags are still on. Can I swap it for a large? No rush."},
 {"id": "S-4", "subject": "Promo code box freezes",
  "body": "On my iPhone the promo code box freezes checkout as soon as I tap it. Works fine on my laptop."},
 {"id": "S-5", "subject": "Love the new tent",
  "body": "Just wanted to say the new tent was easy to set up and kept us dry all weekend. Thanks!"},
])

if USE_SAMPLE:
    items = SAMPLE_ITEMS.copy()
    print("Using the 5 built-in sample items.")
else:
    from google.colab import files
    print("Choose your CSV file...")
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    items = pd.read_csv(io.BytesIO(uploaded[name]))
    if "id" not in items.columns:
        items.insert(0, "id", [f"ITEM-{i+1}" for i in range(len(items))])
    print("Loaded", name, "with", len(items), "rows and columns", list(items.columns))

items = items.head(SAMPLE_SIZE).reset_index(drop=True)

def item_text(row):
    """Everything except the id, shown to the model as 'column: value' lines."""
    return "\n".join(f"{c}: {row[c]}" for c in items.columns if c != "id")

print("\nHere is exactly what the model will be shown for the first item:\n")
print(item_text(items.iloc[0]))

## Part 3 — Run the model on every item
This loop sends one item at a time with `temperature=0` (the most repeatable setting), parses the JSON answer, and records tokens and time. It pauses between calls to stay inside the free-tier rate limit.

**Expect 1–3 minutes for 40 items.** If a few answers cannot be parsed as JSON, that is a finding: count it as an error in your evaluation, and consider whether your output instructions need to be clearer.

In [ ]:
import time, json

PAUSE_SECONDS = 2      # ✏️ raise this if you see rate-limit messages
MAX_TOKENS    = 400    # longest answer the model may give

SYSTEM_PROMPT = "You follow the task instructions exactly and return only what is asked for."

def parse_json(text):
    """Pull the first {...} block out of the model's reply and parse it. Returns (dict or None)."""
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m: return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def run_items(task_description, items_df, label="v1"):
    rows = []
    for i, r in items_df.iterrows():
        prompt = f"{task_description}\n\n--- ITEM ---\n{item_text(r)}\n--- END ITEM ---"
        t0 = time.time()
        resp = ask(prompt, system=SYSTEM_PROMPT, temperature=0.0, max_tokens=MAX_TOKENS)
        secs = time.time() - t0
        raw = resp.choices[0].message.content.strip()
        parsed = parse_json(raw)
        rows.append({"id": r["id"], "item_text": item_text(r).replace("\n", " | "),
                     "model_output": raw.replace("\n", " "), "parsed_ok": parsed is not None,
                     "parsed": json.dumps(parsed) if parsed else "",
                     "tokens_in": resp.usage.prompt_tokens, "tokens_out": resp.usage.completion_tokens,
                     "seconds": round(secs, 2)})
        print(f"[{label}] {i+1}/{len(items_df)}  {r['id']}  ->  {(json.dumps(parsed) if parsed else 'COULD NOT PARSE')[:90]}")
        time.sleep(PAUSE_SECONDS)
    return pd.DataFrame(rows)

results_v1 = run_items(TASK_DESCRIPTION, items, "v1")
n_bad = int((~results_v1["parsed_ok"]).sum())
print(f"\nDone. {len(results_v1)} items run; {n_bad} answers could not be parsed.")

## Part 4 — Pick your 20 and save the labeling sheet
The notebook chooses **20 items at random** (same 20 every time, thanks to `SEED`) and writes `results_v1.csv`. Download it, open it in Google Sheets, and **label each of the 20 rows**:

* `human_correct` — type `1` if the model's output is fully correct, `0` if not. Decide by reading the original item (`item_text`), **not** by trusting the model.
* `error_type` — for each `0`, a short label you invent (for example `sarcasm`, `two issues in one email`, `missing info`, `format`). Reuse labels so you can count them.
* `notes` — write down judgment calls. Where reasonable people could disagree, say so.

In [ ]:
rng = random.Random(SEED)
ids = list(results_v1["id"])
label_ids = set(rng.sample(ids, min(N_TO_LABEL, len(ids))))

sheet = results_v1.copy()
sheet["in_label_sample"] = sheet["id"].isin(label_ids)
sheet["human_correct"] = ""      # you fill in: 1 or 0
sheet["error_type"] = ""
sheet["notes"] = ""
sheet = sheet.sort_values(["in_label_sample", "id"], ascending=[False, True])
sheet.to_csv("results_v1.csv", index=False)

print(f"Saved results_v1.csv with {len(sheet)} rows; {len(label_ids)} are marked in_label_sample = True.")
print("Download it, open in Google Sheets, label the TRUE rows, and re-download as CSV.")
try:
    from google.colab import files
    files.download("results_v1.csv")
except Exception as e:
    print("(download not available here:", type(e).__name__, ")")

## Part 5 — Score your labels
Upload your **labeled** CSV (it must have `id`, `human_correct`, `error_type`). Until you have one, `LABELED_FILE_UPLOAD = False` shows a **demo** with made-up labels so that you can see what the report looks like. **The demo numbers mean nothing.**

The report gives you accuracy with an **interval** (the plausible range). With only 20 labels the range is wide. That is the honest answer to "how accurate is it?" — your memo should say so.

In [ ]:
LABELED_FILE_UPLOAD = False    # ✏️ set to True after you have labeled your sheet

def wilson(k, n, z=1.96):
    """95% interval for a proportion: k correct out of n."""
    if n == 0: return (0, 0)
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0, c-h), min(1, c+h))

if LABELED_FILE_UPLOAD:
    from google.colab import files
    up = files.upload()
    labeled = pd.read_csv(io.BytesIO(up[list(up.keys())[0]]))
    demo = False
else:
    demo = True
    labeled = sheet[sheet["in_label_sample"]].copy().head(20)
    labeled["human_correct"] = [1 if (i % 4) else 0 for i in range(len(labeled))]       # DEMO labels
    labeled["error_type"] = ["" if v else ("sarcasm" if i % 2 else "two issues") for i, v in enumerate(labeled["human_correct"])]

labeled = labeled[labeled["human_correct"].astype(str).str.strip().isin(["0", "1", "0.0", "1.0"])].copy()
labeled["human_correct"] = labeled["human_correct"].astype(float).astype(int)

n = len(labeled); k = int(labeled["human_correct"].sum())
if n == 0:
    raise ValueError("No rows have human_correct = 1 or 0. Fill in that column for the 20 rows marked in_label_sample = TRUE.")
lo, hi = wilson(k, n)
print(("*** DEMO LABELS — NOT REAL RESULTS ***\n" if demo else "") + f"Labeled items: {n}")
print(f"Correct: {k}/{n} = {k/n:.0%}   (95% plausible range for the full set: {lo:.0%} to {hi:.0%})")

errs = labeled[labeled["human_correct"] == 0]["error_type"].fillna("").astype(str).str.strip().str.lower()
errs = errs[errs != ""]
print("\nMost common error types:")
print(errs.value_counts().to_string() if len(errs) else "(none recorded)")
parse_fail = int((~results_v1["parsed_ok"]).sum())
print(f"\nAnswers that could not be parsed (count these as errors too): {parse_fail} of {len(results_v1)}")

## Part 6 — Change ONE thing and re-run
Pick a single change: a clearer rule, a better definition of a hard case, or one worked example added to the task description. Change only that, so you know what caused any difference. **Write your hypothesis first** ("We think adding an example of sarcasm will fix the sarcasm errors"), *then* run.

To save labeling effort, this cell re-runs the same 20 items and compares to version 1:
* If the new output is **identical** to v1, your v1 label carries over.
* If it **changed**, you must label it again (it may have gotten better or worse).

In [ ]:
# ✏️ CHANGE ME — your hypothesis, written BEFORE you run:
HYPOTHESIS = "We think that ____ will ____ because ____."

# ✏️ CHANGE ME — v2 of the task description. Start from the v1 text and change ONE thing.
TASK_DESCRIPTION_V2 = TASK_DESCRIPTION + """

Example of a tricky case: (paste one worked example here, with the correct answer.)
"""

sample_items = items[items["id"].isin(label_ids)].reset_index(drop=True)
results_v2 = run_items(TASK_DESCRIPTION_V2, sample_items, "v2")

merged = results_v2.merge(results_v1[["id", "model_output"]], on="id", suffixes=("", "_v1"))
merged["changed"] = merged["parsed"] != results_v1.set_index("id").loc[merged["id"], "parsed"].values
prev = labeled.set_index("id")[["human_correct", "error_type"]] if "human_correct" in labeled else None
hc, et = [], []
for _, r in merged.iterrows():
    if prev is not None and (not r["changed"]) and r["id"] in prev.index:
        hc.append(int(prev.loc[r["id"], "human_correct"])); et.append(str(prev.loc[r["id"], "error_type"]))
    else:
        hc.append(""); et.append("")      # you fill these in for rows where changed = True
merged["human_correct"] = pd.Series(hc, dtype="object")
merged["error_type"] = pd.Series(et, dtype="object")
merged["notes"] = ""
merged.to_csv("results_v2.csv", index=False)
print(f"v2 changed {int(merged['changed'].sum())} of {len(merged)} outputs.")
print("Open results_v2.csv, hand-label ONLY the rows where changed = True, and upload it in the next cell.")
try:
    from google.colab import files
    files.download("results_v2.csv")
except Exception as e:
    print("(download not available here:", type(e).__name__, ")")

In [ ]:
V2_LABELED_UPLOAD = False    # ✏️ set to True after labeling the changed rows of results_v2.csv

if V2_LABELED_UPLOAD:
    from google.colab import files
    up2 = files.upload()
    v2 = pd.read_csv(io.BytesIO(up2[list(up2.keys())[0]]))
    demo2 = False
else:
    demo2 = True
    v2 = merged.copy()
    v2["human_correct"] = [1 if i % 5 else 0 for i in range(len(v2))]       # DEMO labels

v2 = v2[v2["human_correct"].astype(str).str.strip().isin(["0", "1", "0.0", "1.0"])].copy()
v2["human_correct"] = v2["human_correct"].astype(float).astype(int)
k2, n2 = int(v2["human_correct"].sum()), len(v2)
if n2 == 0:
    raise ValueError("No rows in results_v2.csv have human_correct filled in.")
lo2, hi2 = wilson(k2, n2)
print(("*** DEMO LABELS — NOT REAL RESULTS ***\n" if demo2 else "") + f"Version 1: {k}/{n} = {k/n:.0%}   (range {lo:.0%}-{hi:.0%})")
print(f"Version 2: {k2}/{n2} = {k2/n2:.0%}   (range {lo2:.0%}-{hi2:.0%})")
diff = (k2/n2) - (k/n)
overlap = not (hi2 < lo or hi < lo2)
print(f"\nChange: {diff*100:+.0f} percentage points.")
if overlap:
    print("The plausible ranges overlap, so with only ~20 labels you cannot be sure v2 is truly better or worse.")
    print("Say so in your memo. Do not claim an improvement the data cannot support.")
else:
    print("The ranges do not overlap: the difference is unlikely to be luck.")

## Part 7 — Cost and time for your memo
This turns the token counts into a cost per item and a monthly estimate, then prints the numbers you will paste into the **cost spreadsheet**. A real firm would also pay for **review time on errors**: that is in the spreadsheet, not here.

In [ ]:
tin, tout = results_v1["tokens_in"].sum(), results_v1["tokens_out"].sum()
cost_run = tin/1e6*PRICE_IN_PER_M + tout/1e6*PRICE_OUT_PER_M
per_item = cost_run / len(results_v1)
secs_item = results_v1["seconds"].mean()

print(f"Items run:                {len(results_v1)}")
print(f"Tokens in / out (total):  {tin:,} / {tout:,}")
print(f"Model cost for this run:  ${cost_run:.5f}")
print(f"Model cost per item:      ${per_item:.6f}   (that is ${per_item*1000:.4f} per 1,000 items)")
print(f"Model cost per month at {MONTHLY_VOLUME:,} items: ${per_item*MONTHLY_VOLUME:,.2f}")
print(f"Average seconds per item: {secs_item:.2f}")
print("\nNote how small the model cost is compared with the cost of a person reviewing the mistakes.")
print("That comparison is the heart of your memo.\n")
print("Copy into the cost spreadsheet:  tokens_in/item =", round(tin/len(results_v1)), "| tokens_out/item =", round(tout/len(results_v1)),
      "| model $/item =", round(per_item, 6), "| accuracy (hand-labeled) =", f"{k/n:.2f}")

# Save your settings so the notebook can be reproduced
json.dump({"DATASET": DATASET, "MODEL": MODEL, "SAMPLE_SIZE": SAMPLE_SIZE, "SEED": SEED,
           "TASK_DESCRIPTION": TASK_DESCRIPTION, "TASK_DESCRIPTION_V2": TASK_DESCRIPTION_V2,
           "HYPOTHESIS": HYPOTHESIS}, open("run_config.json", "w"), indent=2)
print("\nSaved run_config.json (hand this in with your notebook).")

---
## ✅ Hand-in checklist
* This notebook, run top to bottom, with your settings (and your hypothesis) in the cells above.
* `results_v1.csv` and `results_v2.csv` with your hand-labels.
* `run_config.json`.
* Your cost spreadsheet filled in with the Part 7 numbers.

## Reflection (put the answers in your memo or AI-use log)
1. What was your hand-labeled accuracy, and how wide was the plausible range? What does that width tell a manager?
2. What was the most common error type, and is it the kind of mistake that is cheap or expensive in your workflow?
3. Did your one change help, hurt, or make no detectable difference? Could it be luck?
4. How many answers could not be parsed, and what would that cost in a real process?
5. If you had to pick a *different* 20 items to label, would you expect the same accuracy? Why or why not?